# 05 · Multi-Agent Systems

Agents that loop over tools, hand work to each other, and the challenges that go with it.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfzalManihar/Agentic-AI-Learning/blob/main/05_multi_agent_systems.ipynb)

**Covered in this notebook**

- Multi Agent Systems

> Part of [Agentic AI Learning](https://github.com/AfzalManihar/Agentic-AI-Learning). Add your own API keys in Colab Secrets (key icon) before running.

# Multi Agent Systems

In [ ]:
pip install langchain-groq langgraph langchain-community ddgs -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from google.colab import userdata

my_api_key = userdata.get('groq_api')

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = 'openai/gpt-oss-20b',
    temperature = 0,
    api_key = my_api_key
)

In [ ]:
# Agent 1: Web Search Tool
# While making any tool for the LLM, make sure to add a description (DOCSTRING)
# so llm can understand what to use this tool is for.
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

def web_search(query: str) -> str:
    """Search the web for latest or factual information"""
    print("🔎 web_search called")
    return search.run(query)

In [ ]:
# Agent 2: Tools -> Multiply , add, divide

def add(a: float, b: float) -> float:
    """Adds two numbers"""
    print("➕ add called:")
    return a+b

def multiply(a: float, b: float) -> float:
    """Multiplies two numbers"""
    print("✖️ multiply called")
    return a*b

def divide(a: float, b: float) -> float:
    """Divides two numbers"""
    print("➗ divide called")
    return a/b

In [ ]:
# Each Specialist gets ONLY its own tools
researcher_llm = llm.bind_tools([web_search])
calculator_llm = llm.bind_tools([add, multiply, divide])

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langchain_core.messages import SystemMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages] #Shared state everyone writes
    next: str # supervisor writes who works next
    steps: int # Safety counter to stop agent going into infinite loops

In [ ]:
def supervisor_node(state: State):
    steps = state.get('steps', 0) # Read steps, if no steps, then default 0

    if steps >= 6:
        print("🛑 step limit reached, ending")
        return {"next": "end", "steps": steps + 1}

    else:
        history = state['messages'] # get the conversation history

    prompt = f"""You are a supervisor managing three specialist agents.

    researcher - can search the web. Use for facts, prices, news, current data.
    calculator - can add, multiply, divide. Use for any arithmetic.
    greeter - responds warmly to greetings and casual conversation.

    Read the conversation so far and decide what must happen NEXT.

    Rules:
    - If the user is greeting you or making casual conversation,
      reply: greeter
    - If a fact is still missing, reply: researcher
    - If a number still needs to be computed, reply: calculator
    - If the user's full request has already been answered, reply: end

    Reply with exactly one word:
    greeter, researcher, calculator, or end.

    Conversation:
    {history}
    """

    decision = llm.invoke(prompt).content.strip().lower()

    print(f"🧠 supervisor (step {steps + 1}) decided: {decision}")

    if decision == 'researcher':
        next_agent = 'researcher'

    elif decision == 'calculator':
        next_agent = 'calculator'

    elif decision == 'greeter':
        next_agent = 'greeter'

    else:
        next_agent = 'end'

    return {
        'next': next_agent,
        'steps': steps + 1
    }

In [ ]:
# Researcher Node
def researcher_node(state:State):
    system = SystemMessage(content=
        "You are the researcher. You can ONLY search the web. "
        "Do not do arithmetic. Find the fact, state it clearly with the number, "
        "and stop. Another agent will do the math."
    )
    response = researcher_llm.invoke([system] + state["messages"])
    return {"messages": [response]}

# Calculator node
def calculator_node(state: State):
    system = SystemMessage(content=
        "You are the calculator. You can ONLY do arithmetic using your tools. "
        "Do not search the web. Use numbers already present in the conversation. "
        "Always call a tool, never compute mentally."
    )
    response = calculator_llm.invoke([system] + state["messages"])
    return {"messages": [response]}

In [ ]:
def greeter_node(state: State):
    system = SystemMessage(content=
        "You are a warm and friendly greeter. "
        "Respond naturally and warmly to greetings and casual conversation. "
        "Keep the response short and friendly. "
        "Do not use any tools."
    )

    response = llm.invoke([system] + state["messages"])

    return {
        "messages": [response]
    }

In [ ]:
# Route after the supervisor decides
def route_after_supervisor(state: State) -> Literal['researcher', 'calculator', 'greeter', END]:

    print('Route After Supervisor: ', state['next'])

    if state['next'] == 'researcher':
        return 'researcher'

    elif state['next'] == 'calculator':
        return 'calculator'

    elif state['next'] == 'greeter':
        return 'greeter'

    elif state['next'] == 'end':
        return END

In [ ]:
# Separate tool node for each specialist

researcher_tool_node = ToolNode([web_search])

calculator_tool_node = ToolNode([add, multiply, divide])

In [ ]:
# After a specialist speaks: either run its tools, or report back to the supervisor
def researcher_should_continue(state: State) -> Literal["researcher_tools", "supervisor"]:
    if state["messages"][-1].tool_calls:
        return "researcher_tools"
    return "supervisor"

def calculator_should_continue(state: State) -> Literal["calculator_tools", "supervisor"]:
    if state["messages"][-1].tool_calls:
        return "calculator_tools"
    return "supervisor"

In [ ]:
builder = StateGraph(State)

builder.add_node('supervisor', supervisor_node)
builder.add_node('researcher', researcher_node)
builder.add_node('calculator', calculator_node)
builder.add_node('greeter', greeter_node)
builder.add_node("researcher_tools", researcher_tool_node)
builder.add_node("calculator_tools", calculator_tool_node)

builder.add_edge(START, 'supervisor')
builder.add_conditional_edges('supervisor', route_after_supervisor)

# researcher inner loop
builder.add_conditional_edges('researcher', researcher_should_continue)
builder.add_edge('researcher_tools', 'researcher')

# calculator inner loop
builder.add_conditional_edges('calculator', calculator_should_continue)
builder.add_edge('calculator_tools', 'calculator')

# greeter -> directly goes to supervisor
builder.add_edge('greeter', 'supervisor')


multi_agent = builder.compile()

In [ ]:
# Runner that prints every node, so the loop is visible
def run(query: str):
    initial_state = {"messages": [("user", query)], "steps": 0}

    print(f"User: {query}\n")
    print("--- STARTING MULTI AGENT LOOP ---")

    for step in multi_agent.stream(initial_state):
        node_name = list(step.keys())[0]
        print(f"📍 Node '{node_name}' finished.")

    print("--- END ---")

    final = multi_agent.invoke(initial_state)
    print("\n🤖 Final Answer:")
    print(final["messages"][-1].content)

In [ ]:
run("Who is the current CEO of Intel?")

User: Who is the current CEO of Intel?

--- STARTING MULTI AGENT LOOP ---
🧠 supervisor (step 1) decided: researcher
Route After Supervisor:  researcher
📍 Node 'supervisor' finished.
📍 Node 'researcher' finished.
🔎 web_search called
📍 Node 'researcher_tools' finished.
📍 Node 'researcher' finished.
🧠 supervisor (step 2) decided: end
Route After Supervisor:  end
📍 Node 'supervisor' finished.
--- END ---
🧠 supervisor (step 1) decided: researcher
Route After Supervisor:  researcher
🔎 web_search called
🧠 supervisor (step 2) decided: end
Route After Supervisor:  end

🤖 Final Answer:
The current CEO of Intel is **Lip‑Bu Tan** (appointed March 18, 2025).


In [ ]:
# 3. TWO HOPS - researcher then calculator
# Supervisor must hand off twice. This is the one that shows the loop.
run("Search the current price of gold per gram in India, then multiply it by 50.")

User: Search the current price of gold per gram in India, then multiply it by 50.

--- STARTING MULTI AGENT LOOP ---
🧠 supervisor (step 1) decided: researcher
Route After Supervisor:  researcher
📍 Node 'supervisor' finished.
📍 Node 'researcher' finished.
🔎 web_search called
📍 Node 'researcher_tools' finished.
📍 Node 'researcher' finished.
🧠 supervisor (step 2) decided: calculator
Route After Supervisor:  calculator
📍 Node 'supervisor' finished.
📍 Node 'calculator' finished.
✖️ multiply called
📍 Node 'calculator_tools' finished.
📍 Node 'calculator' finished.
🧠 supervisor (step 3) decided: end
Route After Supervisor:  end
📍 Node 'supervisor' finished.
--- END ---
🧠 supervisor (step 1) decided: researcher
Route After Supervisor:  researcher
🔎 web_search called
🧠 supervisor (step 2) decided: calculator
Route After Supervisor:  calculator
✖️ multiply called
🧠 supervisor (step 3) decided: end
Route After Supervisor:  end

🤖 Final Answer:
₹660,541.50 (₹13,210.83 × 50).


In [ ]:
# Token Limit -> You have generated too much tokens (content), cannot generate more
# Rate Limit -> You are sending requests too quickly, slow down

In [ ]:
run("what is 3*678")

User: what is 3*678

--- STARTING MULTI AGENT LOOP ---
🧠 supervisor (step 1) decided: calculator
Route After Supervisor:  calculator
📍 Node 'supervisor' finished.
📍 Node 'calculator' finished.
✖️ multiply called
📍 Node 'calculator_tools' finished.
📍 Node 'calculator' finished.
🧠 supervisor (step 2) decided: end
Route After Supervisor:  end
📍 Node 'supervisor' finished.
--- END ---
🧠 supervisor (step 1) decided: calculator
Route After Supervisor:  calculator
✖️ multiply called
🧠 supervisor (step 2) decided: end
Route After Supervisor:  end

🤖 Final Answer:
2034


In [ ]:
run("who is ceo of microsoft")

User: who is ceo of microsoft

--- STARTING MULTI AGENT LOOP ---
🧠 supervisor (step 1) decided: researcher
Route After Supervisor:  researcher
📍 Node 'supervisor' finished.
📍 Node 'researcher' finished.
🔎 web_search called
📍 Node 'researcher_tools' finished.
📍 Node 'researcher' finished.
🧠 supervisor (step 2) decided: end
Route After Supervisor:  end
📍 Node 'supervisor' finished.
--- END ---
🧠 supervisor (step 1) decided: researcher
Route After Supervisor:  researcher
🔎 web_search called
🧠 supervisor (step 2) decided: end
Route After Supervisor:  end

🤖 Final Answer:
Satya Nadella is the CEO of Microsoft.


In [ ]:
run("summarise what it helped you with across both turns.")

User: summarise what it helped you with across both turns.

--- STARTING MULTI AGENT LOOP ---
🧠 supervisor (step 1) decided: researcher
Route After Supervisor:  researcher
📍 Node 'supervisor' finished.
📍 Node 'researcher' finished.
🧠 supervisor (step 2) decided: 
Route After Supervisor:  end
📍 Node 'supervisor' finished.
--- END ---
🧠 supervisor (step 1) decided: researcher
Route After Supervisor:  researcher
🧠 supervisor (step 2) decided: end
Route After Supervisor:  end

🤖 Final Answer:
I’m not sure what “it” refers to. Could you clarify what you’d like summarized?


In [ ]:
# Challenge 2 (Easy)
# Add a third specialist: a greeter_node that uses plain LLM (no tools) and just responds warmly.
# Update the supervisor to route greetings

In [ ]:
# Challenge 3 (Hard)
# Add memory to the multi-agent graph.
# Start a conversation where you first ask a math question,
# then ask a search question, then ask the agent to summarise what it helped you with across both turns.